In [1]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

In [4]:
PLANT = 1
OUT = Path(r"C:\Users\VICTUS\Documents\College\Semester 3\Machine Learning\Solar Power Generation Data\processed")
 
H = 4                              # forecast horizon in 15-min steps (4 = 1 hour ahead)
TEST_START = pd.Timestamp("2020-06-11")
DAY_START, DAY_END = 6, 19         # keep rows whose TARGET time is in daytime (clock-based, no leakage)
STEP = pd.Timedelta(minutes=15)
 
df = pd.read_csv(OUT / f"plant{PLANT}_clean.csv", index_col="DATE_TIME", parse_dates=True)
ac = df["AC_POWER"]

from pathlib import Path

In [5]:
X = pd.DataFrame(index=df.index)
X["y"] = ac.shift(-H)                         # target: AC power H steps ahead
X["target_time"] = df.index + H * STEP
 
X["ac_now"] = ac                              # also the persistence baseline
for k in (1, 2, 4, 8):
    X[f"ac_lag{k}"] = ac.shift(k)
X["ac_same_time_yesterday"] = ac.shift(96 - H)   # value at (target time - 24h); also a baseline
X["ac_roll4"] = ac.rolling(4).mean()             # last 1 hour
X["ac_roll12"] = ac.rolling(12).mean()           # last 3 hours
 
X["irr_now"] = df["IRRADIATION"]
X["irr_lag4"] = df["IRRADIATION"].shift(4)
X["irr_roll4"] = df["IRRADIATION"].rolling(4).mean()
X["amb_now"] = df["AMBIENT_TEMPERATURE"]
X["mod_now"] = df["MODULE_TEMPERATURE"]
 
tod = X["target_time"].dt.hour + X["target_time"].dt.minute / 60
X["tod_sin"] = np.sin(2 * np.pi * tod / 24)
X["tod_cos"] = np.cos(2 * np.pi * tod / 24)

In [6]:
X["irr_target"] = df["IRRADIATION"].shift(-H)
X["amb_target"] = df["AMBIENT_TEMPERATURE"].shift(-H)
 
FEATS_A = ["ac_now", "ac_lag1", "ac_lag2", "ac_lag4", "ac_lag8", "ac_same_time_yesterday",
           "ac_roll4", "ac_roll12", "irr_now", "irr_lag4", "irr_roll4", "amb_now", "mod_now",
           "tod_sin", "tod_cos"]
FEATS_B = FEATS_A + ["irr_target", "amb_target"]

In [7]:
# Chronological split by forecast origin; rows straddling the boundary are dropped
X["split"] = np.where(X.index >= TEST_START, "test",
                      np.where(X["target_time"] < TEST_START, "train", "gap"))
 
# Daytime only (by target time)
X = X[(tod >= DAY_START) & (tod < DAY_END)]
 
# Sanity check: target really is AC power at target_time
chk = X.dropna(subset=["y"]).sample(5, random_state=0)
for t, row in chk.iterrows():
    assert row["y"] == ac.loc[row["target_time"]], "target misaligned"

In [8]:
for name, feats in [("A: lagged only", FEATS_A), ("B: + weather forecast", FEATS_B)]:
    d = X[X["split"].isin(["train", "test"])].dropna(subset=feats + ["y"])
    print(f"\nSetup {name}  ({len(feats)} features)")
    print(d.groupby("split").size().to_string())
    for s in ("train", "test"):
        ds = d[d["split"] == s]
        print(f"  {s}: {ds.index.min()} -> {ds.index.max()}")


Setup A: lagged only  (15 features)
split
test      349
train    1152
  train: 2020-05-16 05:30:00 -> 2020-06-10 17:45:00
  test: 2020-06-11 05:00:00 -> 2020-06-17 17:45:00

Setup B: + weather forecast  (17 features)
split
test      349
train    1152
  train: 2020-05-16 05:30:00 -> 2020-06-10 17:45:00
  test: 2020-06-11 05:00:00 -> 2020-06-17 17:45:00


In [9]:
X.to_csv(OUT / f"plant{PLANT}_features.csv")
with open(OUT / "feature_sets.json", "w") as f:
    json.dump({"A_lagged_only": FEATS_A, "B_with_forecast": FEATS_B}, f, indent=2)
print(f"\nSaved {OUT / f'plant{PLANT}_features.csv'} and feature_sets.json")


Saved C:\Users\VICTUS\Documents\College\Semester 3\Machine Learning\Solar Power Generation Data\processed\plant1_features.csv and feature_sets.json
